# Polynomial Features - Pipeline and Regularization Companion Notebook

This notebook is the practical code companion for the sections of [`lecture_notes/13_polynomial_features.pdf`](../../lecture_notes/13_polynomial_features.pdf) that deal with overfitting, regularization, feature scaling, and the machine-learning pipeline. The central message is that polynomial features should be treated as one component of a full modeling workflow, not as an isolated trick.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42
rng = np.random.default_rng(SEED)


This setup cell imports the objects used throughout the notebook. The experiments are split between regression and classification, but the main workflow principle is the same in both cases: polynomial expansion belongs inside the pipeline together with scaling and the final estimator.


## How Fast Does the Number of Features Grow?

Before looking at model performance, it is worth seeing how the feature space expands as the degree increases.


In [ ]:
rows = []
for n_features in [2, 3, 5, 10]:
    X_dummy = np.ones((1, n_features))
    for degree in [1, 2, 3, 4]:
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        n_out = poly.fit_transform(X_dummy).shape[1]
        rows.append({
            "n_original_features": n_features,
            "degree": degree,
            "n_polynomial_features": n_out,
        })

growth_df = pd.DataFrame(rows)
growth_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for n_features in sorted(growth_df["n_original_features"].unique()):
    dfm = growth_df[growth_df["n_original_features"] == n_features]
    ax.plot(dfm["degree"], dfm["n_polynomial_features"], marker="o", label=f"{n_features} original features")

ax.set_xlabel("polynomial degree")
ax.set_ylabel("number of derived features")
ax.set_title("Polynomial expansion grows quickly with degree and dimension")
ax.legend()
plt.tight_layout()
plt.show()


This first experiment explains why polynomial features can become dangerous. The feature space may explode quickly, especially when the original dimension is already moderate. That extra flexibility is useful, but it also increases the risk of overfitting.


## Degree Selection in Polynomial Regression

We now use validation-style scores to compare polynomial degrees on a regression problem.


In [ ]:
def f(x):
    return np.sin(1.5 * np.pi * x)


X = np.linspace(0, 1, 60)
y = f(X) + rng.normal(0, 0.15, size=X.shape[0])
X = X.reshape(-1, 1)

degree_rows = []
for degree in range(1, 11):
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("linreg", LinearRegression()),
    ])
    train_scores = cross_val_score(model, X, y, cv=5, scoring="neg_mean_squared_error")
    degree_rows.append({
        "degree": degree,
        "cv_mse": -train_scores.mean(),
        "cv_std": train_scores.std(),
    })

degree_df = pd.DataFrame(degree_rows)
degree_df.round(4)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(degree_df["degree"], degree_df["cv_mse"], marker="o")
plt.xlabel("polynomial degree")
plt.ylabel("cross-validated MSE")
plt.title("Model selection should treat the degree as a hyperparameter")
plt.show()


The degree is not something that should be chosen by visual intuition alone. It is a hyperparameter of the pipeline and should be selected using validation or cross-validation, exactly as the lecture notes recommend.


## Why Regularization Matters

A high-degree polynomial can fit training data closely, but that flexibility often needs to be controlled.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_plot = np.linspace(0, 1, 300).reshape(-1, 1)
y_true = f(X_plot.ravel())

ols_high_degree = Pipeline([
        ("poly", PolynomialFeatures(degree=12, include_bias=False)),
        ("linreg", LinearRegression()),
])

ridge_high_degree = Pipeline([
        ("poly", PolynomialFeatures(degree=12, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0)),
])

ols_high_degree.fit(X_train, y_train)
ridge_high_degree.fit(X_train, y_train)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, model, title in [
    (axes[0], ols_high_degree, "Degree-12 polynomial + OLS"),
    (axes[1], ridge_high_degree, "Degree-12 polynomial + Ridge"),
]:
    ax.scatter(X_train, y_train, color="tab:blue", alpha=0.8, label="train")
    ax.plot(X_plot, y_true, color="black", label="true function")
    ax.plot(X_plot, model.predict(X_plot), color="tab:red", label="fitted curve")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.grid(True)
    ax.legend()

axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()


This comparison is the practical reason regularization appears so quickly in the notes. The polynomial expansion can make the model expressive enough to track noise, while Ridge penalizes overly large coefficients and usually leads to a smoother, more stable fit.


In [ ]:
comparison = pd.DataFrame([
    {"model": "Degree-12 OLS", "test_mse": np.mean((y_test - ols_high_degree.predict(X_test)) ** 2)},
    {"model": "Degree-12 Ridge", "test_mse": np.mean((y_test - ridge_high_degree.predict(X_test)) ** 2)},
]).round(4)
comparison


The test-set comparison confirms the geometric picture above. A more flexible model is not automatically better; what matters is how well that flexibility generalizes beyond the training sample.


## Polynomial Features Belong Inside the Pipeline

The final section gives a compact classification example showing the recommended workflow structure.


In [ ]:
X_cls = rng.normal(0, 1, size=(250, 2))
y_cls = ((X_cls[:, 0] ** 2 + X_cls[:, 1] ** 2) < 1.8).astype(int)

rows = []
for degree in [1, 2, 3]:
    pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000)),
    ])
    scores = cross_val_score(pipe, X_cls, y_cls, cv=5, scoring="accuracy")
    rows.append({
        "degree": degree,
        "cv_accuracy": scores.mean(),
    })

pd.DataFrame(rows).round(4)


This final example keeps the workflow compact: polynomial expansion, scaling, and logistic regression are all treated as one pipeline. That is the correct mental model for polynomial features in applied machine learning: they are part of the representation-and-model pipeline, not a standalone preprocessing trick.
